## The Spark API hierarchy

Three structured-data APIs in Spark, in increasing order of abstraction:

- **RDD** — the original. Functional, row-by-row. **No Catalyst optimization, no Tungsten codegen.**
- **DataFrame** — a distributed table with a schema. The everyday API.
- **Dataset[T]** — a typed DataFrame with compile-time type safety. **Scala/Java only.**

In PySpark you only get DataFrames. Internally, `DataFrame` is just an alias for `Dataset[Row]` — same engine, no static type parameter.

The important practical fact: **SQL, DataFrame, and (Scala) Dataset all compile through Catalyst and produce identical execution plans.** RDD bypasses Catalyst — that's why it's slower for structured data.

![Spark API Hierarchy](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-api-hierarchy.svg)

## What a DataFrame really is

A DataFrame is a **distributed table**: one schema (column names + types) shared across the whole dataset, plus rows partitioned across executor memory.

| schema (applies to every partition) |
|:---|
| `transaction_id STRING`, `card_id STRING`, `merchant_category STRING`, `amount DECIMAL(18,2)`, `status STRING` |

| partition 1 — executor 1 | partition 2 — executor 2 |
|---|---|
| T0001, C001, Groceries, 1200.00, APPROVED | T0007, C004, Travel, 42000.00, APPROVED |
| T0002, C002, Travel, 18500.00, APPROVED | T0008, C003, Groceries, 1850.00, APPROVED |
| T0003, C001, Food, 450.00, APPROVED | T0009, C005, Shopping, 3200.00, APPROVED |

Each partition holds a **subset of rows with all columns** — partitioning is always horizontal (by row), never vertical (by column).

Properties:

- **Immutable** — every transformation returns a new DataFrame.
- **Lazy** — transformations build a plan; actions execute it.
- **Schema-enforced** — every row matches the same column types.
- **Partitioned** — rows split across partitions for parallel processing.

## SparkSession — the unified entry point

`SparkSession` was introduced in Spark 2.0 to merge three older entry points (`SQLContext`, `HiveContext`) into one object, while still wrapping the underlying `SparkContext`.

| Entry point | What you get |
|---|---|
| `spark.read` | `DataFrameReader` — load files, JDBC, Delta |
| `spark.sql("...")` | Run a SQL string, returns a DataFrame |
| `spark.range(n)` | DataFrame of integers `0` to `n-1` |
| `spark.createDataFrame(data, schema)` | Build a DataFrame from a Python list or pandas DataFrame |
| `spark.catalog` | Inspect / manage tables, databases, functions |
| `spark.conf` | Read / set runtime configuration |
| `spark.sparkContext` | Drop down to the RDD-level API |

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("DataFramesExecutionModel")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("version            :", spark.version)
print("AQE enabled        :", spark.conf.get("spark.sql.adaptive.enabled"))
print("shuffle partitions :", spark.conf.get("spark.sql.shuffle.partitions"))
print("databases          :", [db.name for db in spark.catalog.listDatabases()])

## Schemas — `StructType` vs DDL string

Two ways to define a schema. Both produce the exact same result; pick the one that reads better in your code.

- **Programmatic** — `StructType([StructField(...)])`. Verbose, explicit, easy to build from code.
- **DDL string** — `"col_a STRING, col_b DECIMAL(18,2), col_c TIMESTAMP"`. Compact, SQL-flavored, easy to paste.

**Always prefer explicit schemas in production.** Schema inference works but it triggers a full scan of the data to decide types — slow on large datasets, and it can guess wrong (e.g., a sparse `is_flagged` column inferred as `string` when you wanted `boolean`).

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DecimalType, BooleanType
)

# Programmatic — matches project/schemas/bank_schemas.py
txn_schema_struct = StructType([
    StructField("transaction_id",    StringType(),       nullable=False),
    StructField("card_id",           StringType(),       nullable=False),
    StructField("customer_id",       StringType(),       nullable=False),
    StructField("merchant_category", StringType()),
    StructField("amount",            DecimalType(18, 2), nullable=False),
    StructField("status",            StringType()),
    StructField("is_flagged",        BooleanType()),
])

# DDL string — same schema, more compact
txn_schema_ddl = (
    "transaction_id STRING NOT NULL, card_id STRING NOT NULL, customer_id STRING NOT NULL, "
    "merchant_category STRING, amount DECIMAL(18,2) NOT NULL, status STRING, is_flagged BOOLEAN"
)

print(txn_schema_struct.simpleString())

## Creating DataFrames

Five common ways. The first three are workhorses for tests; the last two are everyday.

- `spark.createDataFrame(data, schema)` — Python list + explicit schema (preferred).
- `spark.createDataFrame(data)` — Python list + schema inference.
- `spark.range(n)` — fast integer-sequence DataFrame for testing.
- `spark.createDataFrame(pandas_df)` — round-trip from pandas.
- `spark.read.format(...).load(path)` — from real files (covered in notebook 04).

In [ ]:
from decimal import Decimal

# 1. Python list with explicit schema (preferred)
data = [
    ("T0001", "C001", "CUST001", "Groceries",     Decimal("1200.00"),  "APPROVED", False),
    ("T0002", "C002", "CUST002", "Travel",        Decimal("18500.00"), "APPROVED", False),
    ("T0003", "C001", "CUST001", "Food",          Decimal("450.00"),   "APPROVED", False),
    ("T0004", "C003", "CUST003", "Shopping",      Decimal("6700.00"),  "APPROVED", True),
    ("T0005", "C002", "CUST002", "Fuel",          Decimal("2800.00"),  "DECLINED", False),
    ("T0006", "C001", "CUST001", "Entertainment", Decimal("1100.00"),  "APPROVED", False),
    ("T0007", "C004", "CUST004", "Travel",        Decimal("42000.00"), "APPROVED", True),
    ("T0008", "C003", "CUST003", "Groceries",     Decimal("1850.00"),  "APPROVED", False),
]
df = spark.createDataFrame(data, schema=txn_schema_struct)

# 2. Schema inference — convenient but slow on large data
df_inferred = spark.createDataFrame(
    data,
    ["transaction_id", "card_id", "customer_id", "merchant_category",
     "amount", "status", "is_flagged"],
)

# 3. spark.range — fast integer sequence
df_range = spark.range(0, 5).toDF("idx")

# 4. From a pandas DataFrame
import pandas as pd
pdf = pd.DataFrame({"category": ["Food", "Travel"], "weight": [0.3, 0.7]})
df_from_pandas = spark.createDataFrame(pdf)

print("explicit  :", df.count(), "rows")
print("inferred  :", df_inferred.count(), "rows")
print("range     :", df_range.count(), "rows")
print("pandas    :", df_from_pandas.count(), "rows")

## Exploring a DataFrame

The cheatsheet for *what does this DataFrame contain?*

| Method | Returns |
|---|---|
| `df.count()` | Number of rows (action — triggers a job) |
| `df.columns` | Column names as a Python list |
| `df.dtypes` | List of `(name, type_string)` tuples |
| `df.schema` | The full `StructType` object |
| `df.printSchema()` | Pretty-print the schema |
| `df.describe()` | Count, mean, stddev, min, max for numeric columns |
| `df.summary()` | `describe()` + quartiles |
| `df.show(n, truncate=False)` | Print rows; pass `truncate=False` for full content |

In [ ]:
print("rows x cols :", df.count(), "x", len(df.columns))
print("columns     :", df.columns)
print("dtypes      :", df.dtypes)
print()
df.printSchema()
print()
df.show(3, truncate=False)
print()
df.select("amount").describe().show()

## Column references — the four ways

PySpark accepts four styles for naming a column. They are *mostly* interchangeable but each has a place.

| Form | When it shines | When it breaks |
|---|---|---|
| `"col_name"` (string) | Simple `select`, `groupBy`, `sort` | Cannot apply functions or build expressions |
| `col("col_name")` | Building expressions; passing to `when`, `coalesce`, `F.sum` | None — this is the safest default |
| `df["col_name"]` | Disambiguating columns from two DataFrames in a join | Verbose for normal use |
| `expr("col_name * 0.02")` | Pasting a SQL-style snippet | Easy to typo; the string is parsed at runtime |

Rule of thumb: use a **string** when the API just needs a name, **`col()`** when building an expression, **`df["c"]`** when joining, and **`expr()`** sparingly.

In [ ]:
from pyspark.sql.functions import col, expr

# All four below produce the exact same projected DataFrame.
df.select("merchant_category", "amount").show(3)
df.select(col("merchant_category"), col("amount")).show(3)
df.select(df["merchant_category"], df["amount"]).show(3)
df.selectExpr("merchant_category", "amount").show(3)

# Where col() actually matters — building expressions
df.select(col("amount"), (col("amount") * 0.02).alias("processing_fee")).show(3)

## Common DataFrame operations

The day-to-day toolkit. One cell, five families:

- **Select / project** — `select`, `selectExpr`
- **Add / rename / drop** — `withColumn`, `withColumnRenamed`, `drop`
- **Filter** — `filter` ≡ `where`; combine conditions with `&`, `|`, `~`; **parenthesize each condition** (Python operator precedence will mislead you otherwise)
- **Nulls** — `isNull`, `isNotNull`, `dropna`, `fillna`, `coalesce(...)`
- **Cast / sort / dedupe** — `.cast()`, `orderBy(desc(...))`, `distinct()` vs `dropDuplicates(["col"])`

In [ ]:
from pyspark.sql.functions import col, lit, when, upper, coalesce, desc
from pyspark.sql.types import DoubleType

# --- Select / project -------------------------------------------------------
df.select("transaction_id", "merchant_category", "amount").show(3)
df.selectExpr("transaction_id", "merchant_category", "amount * 0.02 AS fee").show(3)

# --- Add / rename / drop ----------------------------------------------------
df_with = (
    df
    .withColumn("amount_double", col("amount").cast(DoubleType()))                      # add new
    .withColumn("status",        upper(col("status")))                                  # replace existing
    .withColumn("tier",          when(col("amount") > 10000, "HIGH").otherwise("STANDARD"))
    .withColumnRenamed("merchant_category", "category")
    .drop("is_flagged")
)
df_with.show(3, truncate=False)

# --- Filter ------------------------------------------------------------------
# Wrap each condition in (); &, |, ~ are bitwise operators with low precedence in Python.
df.filter((col("status") == "APPROVED") & (col("amount") > 1000)).show(3)
df.where(col("status") != "APPROVED").show(3)
df.filter(col("merchant_category").isin("Travel", "Shopping")).show(3)

# --- Nulls -------------------------------------------------------------------
df_nullable = df.withColumn(
    "merchant_category",
    when(col("transaction_id") == "T0005", lit(None)).otherwise(col("merchant_category")),
)
df_nullable.filter(col("merchant_category").isNull()).show()
df_nullable.fillna({"merchant_category": "UNKNOWN"}).show(3)
df_nullable.select(
    coalesce(col("merchant_category"), lit("UNKNOWN")).alias("category")
).show(3)

# --- Cast / sort / dedupe ---------------------------------------------------
df.select(col("amount").cast(DoubleType()).alias("amount_double")).show(3)
df.orderBy(desc("amount")).show(3)
df.select("merchant_category").distinct().show()                  # all-column distinct
df.dropDuplicates(["card_id"]).show(3)                            # keyed dedupe

## DataFrame ↔ pandas / RDD interop

Three escape hatches you should know.

- **`df.toPandas()`** — pulls every row into a single pandas DataFrame *on the driver*. Convenient for plotting and final-mile analysis. **Dangerous on large data — it OOMs the driver.** Always sample or aggregate first.
- **`spark.createDataFrame(pandas_df)`** — the reverse direction; useful for tiny test fixtures.
- **`df.rdd`** — drop down to an RDD of `Row` objects when you need RDD-only operations like `mapPartitions`. The reverse round-trip is `rdd.toDF(schema)`.

In [ ]:
# To pandas — only on small results
small_pdf = df.filter(col("status") == "APPROVED").limit(3).toPandas()
print(type(small_pdf))
print(small_pdf)
print()

# To RDD — element type is Row
first_row = df.rdd.first()
print(type(first_row))
print("merchant_category :", first_row["merchant_category"])
print("amount            :", first_row.amount)

# Execution Model

Everything above was *what a DataFrame is and how to manipulate it*. The rest of this notebook is *what happens when you call an action*: how Spark turns your DataFrame chain into work that actually runs on a cluster.

Five layers, top to bottom:

- **Lazy evaluation** — why nothing runs until an action fires, and what Spark gains by waiting.
- **Catalyst** — the rule-based optimizer that rewrites your plan in four stages before any executor sees it.
- **Tungsten** — the runtime layer that makes the optimized plan fast on the JVM (off-heap memory, whole-stage codegen).
- **`.explain()`** — the window into all of the above.
- **Adaptive Query Execution (AQE)** — runtime re-optimization that adjusts the plan *after* shuffles, using real statistics instead of estimates.

The thread tying these together: **Catalyst can only optimize what it can see, and it can only see the whole pipeline because nothing has run yet.** Lazy evaluation is the precondition; Catalyst, Tungsten, and AQE are what Spark does with that visibility.

## Lazy evaluation, in depth

Notebook 01 introduced the rule: transformations are recipes, actions trigger cooking. Here is the *why* in one sentence: **Catalyst can only optimize what it can see, and it can only see the whole pipeline because nothing has run yet.**

DataFrame edition of the table:

| | Transformations | Actions |
|---|---|---|
| Execution | Lazy — append to the plan | Eager — trigger DAG execution |
| Returns | Another DataFrame | A value, file, or printed output |
| Examples | `select`, `filter`, `withColumn`, `groupBy`, `join`, `orderBy` | `show`, `count`, `collect`, `take`, `first`, `write` |

Because it's lazy, Spark will:

- **Push filters early** — a filter declared at the end of the chain may execute right next to the data scan.
- **Prune projections** — columns you never reference are dropped before reading.
- **Fuse operations** — multiple narrow transformations collapse into one tight loop (whole-stage codegen, see §12).
- **Skip everything** — if you define transformations and never call an action, no work runs.

## The four Catalyst plans

When you call an action, your DataFrame chain travels through Catalyst as four distinct plans before hitting the cluster.

1. **Parsed logical plan** — your code as a tree of unresolved references. Column names are still strings; types are unknown.
2. **Analyzed logical plan** — Catalyst resolves column names against the schema, checks types, and binds function calls. Errors here look like `AnalysisException`.
3. **Optimized logical plan** — rule-based rewrites: predicate pushdown (filters move toward the scan), projection pruning (drop unused columns), constant folding (`1 + 2 → 3`), reordering, eliminating redundant projects.
4. **Physical plan** — Catalyst picks operators (`HashAggregate` vs `SortAggregate`, `BroadcastHashJoin` vs `SortMergeJoin`) and a physical strategy. The output is what actually runs on executors.

![Catalyst Phases](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-catalyst-phases.svg)

You can inspect each plan with `.explain(extended=True)` — we'll do that in §13.

## Tungsten — making the JVM go fast

Catalyst gives you the right *plan*. **Tungsten** makes that plan run fast on the JVM. Two ideas:

- **Off-heap memory management** — Spark allocates structured binary buffers off the JVM heap, bypassing the garbage collector. Less GC pause, more cache-friendly memory layouts.
- **Whole-stage code generation (WSCG)** — for each stage, Catalyst generates a single tightly-fused JVM method that processes a row through every operator at once (filter → project → aggregate). One loop instead of one method call per operator per row.

This is why DataFrames beat RDDs on structured data: RDDs have no schema, so Tungsten cannot lay them out efficiently or generate per-stage code. With DataFrames, the schema is a contract Tungsten can compile against.

You will see the proof in the next section: the physical plan for a DataFrame query is wrapped in a `*(N) WholeStageCodegen` block.

## `.explain()` modes — seeing inside Catalyst

`df.explain(mode="...")` exposes the plans Catalyst built. Five modes:

| Mode | Shows |
|---|---|
| `"simple"` (default) | Just the physical plan |
| `"extended"` | All four plans (parsed → analyzed → optimized → physical) |
| `"codegen"` | The generated JVM code per stage |
| `"cost"` | Optimized plan with row-count estimates per node |
| `"formatted"` | Physical plan + a separate operator-details section |

Same query, multiple lenses.

In [ ]:
from pyspark.sql.functions import sum as spark_sum

# A small but representative query: filter, group, aggregate.
query = (
    df
    .filter(col("status") == "APPROVED")
    .groupBy("merchant_category")
    .agg(spark_sum("amount").alias("total_amount"))
)

print("=" * 70)
print("simple — physical plan only")
print("=" * 70)
query.explain()

In [ ]:
print("=" * 70)
print("extended — all four plans")
print("=" * 70)
query.explain(extended=True)

In [ ]:
print("=" * 70)
print("formatted — physical plan + operator details")
print("=" * 70)
query.explain(mode="formatted")

## Adaptive Query Execution (AQE)

Before Spark 3.0, Catalyst made all decisions upfront using *estimated* statistics. **AQE re-optimizes the plan at runtime**, after each shuffle, using *real* partition statistics.

Three things AQE actually does:

- **Dynamic coalescing** — after a shuffle produces, say, 200 partitions but most are tiny, AQE merges them into a smaller number of right-sized partitions. Fewer task-startup overheads.
- **Dynamic join strategy switch** — a sort-merge join can be promoted to a broadcast join at runtime if one side turns out small enough.
- **Skew join handling** — splits oversized partitions so one slow task doesn't block a stage. *Covered in detail in notebook 07 (Performance & Tuning).*

AQE is enabled by default since Spark 3.2. Below we run the same `groupBy` aggregation with AQE on and off to see the post-shuffle partition count change.

In [ ]:
# AQE on: post-shuffle coalescing reduces the partition count.
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("AQE on  : output partitions =", query.rdd.getNumPartitions())

# AQE off: post-shuffle stays at spark.sql.shuffle.partitions (default 200).
spark.conf.set("spark.sql.adaptive.enabled", "false")
query_no_aqe = (
    df
    .filter(col("status") == "APPROVED")
    .groupBy("merchant_category")
    .agg(spark_sum("amount").alias("total_amount"))
)
print("AQE off : output partitions =", query_no_aqe.rdd.getNumPartitions())

# Re-enable AQE for any later cells.
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [ ]:
spark.stop()